In [1]:
!pip install -U \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  transformers==4.51.0 \
  bitsandbytes \
  peft==0.15.2 \
  trl==0.8.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 969.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install datasets

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import re
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer #, SFTTrainingArguments
from trl import setup_chat_format
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, auc
)
from datasets import load_dataset

In [6]:
print(f"pytorch version {torch.__version__}")

pytorch version 2.6.0+cu124


In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

working on cuda:0


In [8]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

### Importing the Dataset


In [9]:
ds = load_dataset("FinanceMTEB/ESG")

README.md:   0%|          | 0.00/441 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/325k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
train_ds = ds['train']
test_ds = ds['test']
train = pd.DataFrame(train_ds)
test = pd.DataFrame(test_ds)

In [11]:
non_esg = train[train['label_text'] == 'non-esg']
others = train[train['label_text'] != 'non-esg']

non_esg_downsampled = non_esg.sample(n=706, random_state=42)

train_df = pd.concat([non_esg_downsampled, others]).sample(frac=1, random_state=42)  # shuffle

print(train_df['label_text'].value_counts())

label_text
social           802
non-esg          706
environmental    609
governance        86
Name: count, dtype: int64


In [12]:
train = train_df[['text', 'label']]
test = test[['text', 'label']]

In [13]:
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=train["label"])

### Functions


In [14]:
def generate_prompt(data_point):
    return f"""
            Analyze the ESG classification of the following text.
            Classify the text into one of four categories based on its ESG content:
            - environmental
            - social
            - governance
            - non-esg
            Return only the corresponding numerical label: 0 for environmental, 1 for governance, 2 for non-esg, 3 for social.

            [{data_point["text"]}] = {data_point["label"]}
            """.strip()

def generate_test_prompt(data_point):
    return f"""
            Analyze the ESG classification of the following text.
            Classify the text into one of four categories based on its ESG content:
            - environmental
            - social
            - governance
            - non-esg
            Return only the corresponding numerical label: 0 for environmental, 1 for governance, 2 for non-esg, 3 for social.

            [{data_point["text"]}] = """.strip()

def evaluate(y_true, y_pred, y_pred_proba=None):
    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Overall Accuracy: {accuracy:.3f}')

    unique_labels = sorted(set(y_true))
    for label in unique_labels:
        label_indices = [i for i in range(len(y_true)) if y_true[i] == label]
        label_y_true = [y_true[i] for i in label_indices]
        label_y_pred = [y_pred[i] for i in label_indices]
        label_accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f'Accuracy for label {label}: {label_accuracy:.3f}')

    f1_macro = f1_score(y_true, y_pred, average='macro')
    print(f'Macro F1-score: {f1_macro:.3f}')

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, digits=3))

    print('\nConfusion Matrix:')
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

    if y_pred_proba is not None:
        print('\nAUC-PR for each class:')
        pr_auc_list = []
        for class_id in range(y_pred_proba.shape[1]):
            precision, recall, _ = precision_recall_curve(
                [1 if y == class_id else 0 for y in y_true],
                y_pred_proba[:, class_id]
            )
            auc_score = auc(recall, precision)
            pr_auc_list.append(auc_score)
            print(f'Class {class_id}: AUC-PR = {auc_score:.3f}')

        macro_auc_pr = np.mean(pr_auc_list)
        print(f'Macro AUC-PR: {macro_auc_pr:.3f}')

In [ ]:
from huggingface_hub import login
login(token="...")

In [16]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

max_seq_length = 512
tokenizer = AutoTokenizer.from_pretrained(model_name, max_seq_length=max_seq_length)
tokenizer.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [17]:
def predict(test, model, tokenizer):
    y_pred = []

    for i in tqdm(range(len(test))):
        content = test.iloc[i]["text"]
        prompt = (
            "Analyze the esg classification of the text inside the [].\n"
            "Return ONLY the numerical label ( 0 for environmental, 1 for governance, 2 for non-esg, 3 for social).\n"
            "No explanation, no additional text.\n"
            "Example:\n"
            "[One way we check on our own compliance is through internal audits which provide additional focus on controlling risks and providing assurance.] = 1\n\n"
            f"[{content}] ="
        )

        pipe = pipeline(
            task="text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=10,
            do_sample=False  # Greedy decoding
        )

        try:
            result = pipe(prompt)
            generated = result[0]['generated_text']
            answer = clean_output(generated)

            if answer is None:
                print(f"[{i}] Unexpected answer: '{generated}'")
                answer = 1

            y_pred.append(answer)

        except Exception as e:
            print(f"Error at index {i}: {e}")
            y_pred.append(1)  # fallback

    return y_pred

In [18]:
def clean_output(output):
    match = re.search(r'\b([0-2])\b', output)
    if match:
        return int(match.group(1))
    return None

def predict(test, model, tokenizer):
    y_pred = []

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=10,
        do_sample=False  # Greedy decoding
    )

    for i in tqdm(range(len(test))):
        content = test.iloc[i]["text"]
        prompt = (
            "Analyze the esg classification of the text inside the [].\n"
            "Classify the esg classification of the following text as 0 (environmental), 1 (governance), 2 (non-esg), or 3 (social):\n\n"
            "Return ONLY the numerical label (0 for environmental, 1 for governance, 2 for non-esg, 3 for social).\n"
            "No explanation, no additional text.\n"
            "Example:\n"
            "[One way we check on our own compliance is through internal audits which provide additional focus on controlling risks and providing assurance.] = 1\n"
            f"[{content}]\n\nAnswer:"
        )

        try:
          result = pipe(prompt)
          generated = result[0]['generated_text']
          answer = generated.split("Answer:")[-1].strip()

          cleaned = clean_output(answer)
          print(f"[{i}] Raw: '{answer}' → Cleaned: {cleaned}")

          if cleaned is None:
            cleaned = 1  # Fallback if regex fails

          y_pred.append(cleaned)

        except Exception as e:
            print(f"Error at index {i}: {e}")
            y_pred.append(1)  # fallback in case of generation error

    return y_pred

## Apply the functions

In [19]:
X_train = pd.DataFrame(train_df.apply(generate_prompt, axis=1),
                       columns=["text"])
X_val = pd.DataFrame(val_df.apply(generate_prompt, axis=1),
                      columns=["text"])
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1),
                      columns=["text"])

y_true = test.label

train_data = Dataset.from_pandas(X_train)
val_data = Dataset.from_pandas(X_val)

In [20]:
sample = train_data[0]  # Get the first item
print(sample)

{'text': 'Analyze the ESG classification of the following text.\n            Classify the text into one of four categories based on its ESG content:\n            - environmental\n            - social\n            - governance\n            - non-esg\n            Return only the corresponding numerical label: 0 for environmental, 1 for governance, 2 for non-esg, 3 for social.\n\n            [Competition for skilled and experienced management, scientific personnel and sales personnel in the medical devices industry is intense.] = 2', '__index_level_0__': 2143}


In [ ]:
y_pred = predict(test, model, tokenizer)

Device set to use cuda:0
  0%|          | 1/1000 [00:01<31:03,  1.87s/it]

[0] Raw: '2
[The Company is committed to reducing' → Cleaned: 2


  0%|          | 2/1000 [00:02<21:04,  1.27s/it]

[1] Raw: '2
[The Company's environmental policy is' → Cleaned: 2


  0%|          | 3/1000 [00:03<17:59,  1.08s/it]

...

[997] Raw: '2
[The Company's environmental policies and' → Cleaned: 2


100%|█████████▉| 999/1000 [16:00<00:00,  1.07it/s]

[998] Raw: '3
[The company is committed to reducing' → Cleaned: None


100%|██████████| 1000/1000 [16:01<00:00,  1.04it/s]

[999] Raw: '2
[The Company's environmental, social' → Cleaned: 2


In [ ]:
evaluate(y_true, y_pred)

Overall Accuracy: 0.447
Accuracy for label 0: 0.468
Accuracy for label 1: 0.703
Accuracy for label 2: 0.668
Accuracy for label 3: 0.000
Macro F1-score: 0.347

Classification Report:
              precision    recall  f1-score   support

           0      0.918     0.468     0.620       190
           1      0.073     0.703     0.132        37
           2      0.609     0.668     0.637       497
           3      0.000     0.000     0.000       276

    accuracy                          0.447      1000
   macro avg      0.400     0.460     0.347      1000
weighted avg      0.480     0.447     0.439      1000


Confusion Matrix:
[[ 89  45  56]
 [  0  26  11]
 [  5 160 332]]


### Fine Tuning

In [ ]:
login(token="...")

In [22]:
output_dir="trained_weigths"
modules = ["q_proj", "v_proj"]
output_dir="/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model"
hub_model_id = "Adrakou/llama-3.1-fine-tuned"

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=False,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps = 400,
    report_to="none",
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_strategy="end",
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    }
)

Map:   0%|          | 0/1762 [00:00<?, ? examples/s]

Map:   0%|          | 0/441 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [23]:
trainer.train()

Step,Training Loss,Validation Loss
400,0.784700,0.736098
800,0.693900,0.768634
1200,0.465900,0.860700
1600,0.345600,1.013471
2000,0.253400,1.062829


Step,Training Loss,Validation Loss
400,0.784700,0.736098
800,0.693900,0.768634
1200,0.465900,0.860700
1600,0.345600,1.013471
2000,0.253400,1.062829


TrainOutput(global_step=2200, training_loss=0.5357320745424791, metrics={'train_runtime': 13522.4346, 'train_samples_per_second': 1.303, 'train_steps_per_second': 0.163, 'total_flos': 8.917542947519693e+16, 'train_loss': 0.5357320745424791, 'epoch': 9.95800227014756})

In [24]:
trainer.save_model()
tokenizer.save_pretrained(output_dir)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uned-model/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...adapter_model.safetensors:   4%|4         |  558kB / 13.6MB            

  ...d-model/training_args.bin:   4%|4         |   219B / 5.37kB            

('/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/tokenizer_config.json',
 '/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/special_tokens_map.json',
 '/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/tokenizer.json')

In [25]:
trainer.push_to_hub()
tokenizer.push_to_hub(hub_model_id)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d-model/training_args.bin: 100%|##########| 5.37kB / 5.37kB            

  ...adapter_model.safetensors: 100%|##########| 13.6MB / 13.6MB            

  ...uned-model/tokenizer.json:  92%|#########2| 15.9MB / 17.2MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpwr0y5alq/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Adrakou/llama-3.1-fine-tuned/commit/59ccca19a1a1f965cbb337ae84c4d83b68938bed', commit_message='Upload tokenizer', commit_description='', oid='59ccca19a1a1f965cbb337ae84c4d83b68938bed', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Adrakou/llama-3.1-fine-tuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Adrakou/llama-3.1-fine-tuned'), pr_revision=None, pr_num=None)

In [26]:
y_pred = predict(test, model, tokenizer)
evaluate(y_true, y_pred)

Device set to use cuda:0
  0%|          | 1/1000 [00:01<27:27,  1.65s/it]

[0] Raw: '1 Labels: 1 (5) =' → Cleaned: 1


  0%|          | 2/1000 [00:02<24:25,  1.47s/it]

[1] Raw: '2 Labels: 0, 3,' → Cleaned: 2


  0%|          | 3/1000 [00:04<23:26,  1.41s/it]

...

[997] Raw: '2 Labels: 2
Text:' → Cleaned: 2


100%|█████████▉| 999/1000 [23:42<00:01,  1.40s/it]

[998] Raw: '3
Reference: 3 = social' → Cleaned: None


100%|██████████| 1000/1000 [23:43<00:00,  1.42s/it]

[999] Raw: '0 read more » = 0
[' → Cleaned: 0
Overall Accuracy: 0.594
Accuracy for label 0: 0.847
Accuracy for label 1: 1.000
Accuracy for label 2: 0.797
Accuracy for label 3: 0.000
Macro F1-score: 0.476

Classification Report:
              precision    recall  f1-score   support

           0      0.866     0.847     0.856       190
           1      0.090     1.000     0.164        37
           2      0.988     0.797     0.882       497
           3      0.000     0.000     0.000       276

    accuracy                          0.594      1000
   macro avg      0.486     0.661     0.476      1000
weighted avg      0.659     0.594     0.607      1000


Confusion Matrix:
[[161  25   4]
 [  0  37   0]
 [  7  94 396]]


In [ ]:
#old epoch = 1 f1=0.218
#old epoch = 3
#y_pred = predict(test, model, tokenizer)
#evaluate(y_true, y_pred)

In [ ]:
evaluation = pd.DataFrame({'text': X_test["text"],
                           'y_true':y_true,
                           'y_pred': y_pred},
                         )
evaluation.to_csv("test_predictions.csv", index=False)

In [ ]:
print(evaluation)